# Sesión 06 laboratorio práctico: Reto de regresión para el precio esperado de mercado

**Rol de canalización:** Laboratorio de aula de la Sesión 06 opcional, fuera de la ruta de ejecución obligatoria de un extremo a otro.

> Advertencia: El cuaderno de referencia de producción sigue siendo `04_regression_audi_a3_germany.ipynb`.

Esta práctica de laboratorio se ejecuta directamente desde archivos CSV procesados. No requiere credenciales en la nube ni acceso al almacén.

**Consume:** archivos CSV procesados ​​con las columnas de regresión requeridas.
**Produce:** resultados del aula y archivos CSV de cola de revisión opcionales bajo `reports/`.
**Feeds:** interpretación en el aula y discusión de transferencia BI, no la tabla de predicción obligatoria BigQuery.

Head Of Data 101 utiliza una idea central: **el canal es el producto**.

En esta práctica de laboratorio, la regresión crea una capa de referencia del precio de mercado esperado para el análisis de adquisición de vehículos usados. La salida del modelo es `expected_price_eur`. La señal comercial es `expected_price_gap = actual price - expected price`.

Una brecha negativa significa que la cotización es más barata de lo que espera el modelo. Una brecha positiva significa que la cotización es más cara de lo que espera el modelo. La brecha no es una recomendación de inversión automática; es una señal de priorización para la revisión de los analistas.

Límite narrativo del curso:

- La regresión predice `expected_price_eur`.
- La regresión produce `expected_price_gap`.
- La clasificación permanece separada y predice `top_price`.
- BI luego combina el precio real, la brecha expected-price y la probabilidad top-price.


## Agenda de sesión en vivo de 3 horas

- 00:00-00:15 - Contrato de datos y objetivo comercial
- 00:15-00:35 - Lectura visual del mercado.
- 00:35-01:05 - Línea base de regresión simple
- 01:05-01:45 - Modelo multivariado expected-price
- 01:45-02:15 - Diagnóstico e interpretación del modelo.
- 02:15-02:45 - Del precio esperado a la cola de revisión
- 02:45-03:00 - Discusión y entrega BI


## Ruta en vivo mínima viable

Si el tiempo de clase se vuelve escaso, complete el Desafío 0, el Desafío 1, el Desafío 2, el Desafío 3 y el Desafío 4; Deje los modelos flexibles opcionales para después de clase.


## Cómo trabajar

Este no es un curso puro de codificación. Utilice la codificación asistida por IA cuando sea útil, pero asegúrese de poder explicar cada resultado.

Para cada desafío, lea el motivo comercial, complete la pequeña tarea de codificación o inspeccione el resultado dirigido por el instructor y escriba una breve interpretación comercial en inglés.

Busque las etiquetas **Tarea del estudiante**, **Demostración dirigida por un instructor**, **Pausa de discusión** y **Extensión opcional**. Están ahí para proteger el ritmo de clase.


## Desafío 0: contrato de datos y conjunto de datos de modelado

**Tiempo sugerido:** 15 minutos

**Demostración dirigida por un instructor**

**Razón comercial:**  
Antes de modelar, un científico de datos profesional debe saber exactamente qué contrato de conjunto de datos se está consumiendo. Este no es un ejercicio de manejo del camino; es un ejercicio de concientización sobre los contratos.

**Objetivo:**  
Cargue un CSV procesado, valide las columnas de regresión requeridas, cree `listing_id` si es necesario y prepare un marco de datos de modelado limpio llamado `df`.

**Columnas obligatorias:** `price_eur`, `mileage_km`, `age_years`, `power_hp`.

**Formato de salida esperado:**
- Ruta CSV seleccionada.
- Recuento de filas antes y después del filtrado.
- Columnas disponibles.
- Primeras 5 filas.
- Resumen de valores faltantes.

**Pausa de discusión:**
- ¿Qué CSV cargó el notebook?
- ¿Por qué las columnas requeridas deberían fallar ruidosamente en lugar de ser adivinadas?


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.style.use("default")


def find_repo_root(start):
    for path in [start] + list(start.parents):
        if (path / ".git").exists() or (path / "config" / "project_config.yaml").exists():
            return path
    return start


def load_project_config(config_path):
    config = {}
    if not config_path.exists():
        return config
    for raw_line in config_path.read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        key, value = line.split(":", 1)
        key = key.strip()
        value = value.strip()
        if value.startswith(("'", '"')) and value.endswith(("'", '"')):
            value = value[1:-1]
        elif value.lower() in ("true", "false"):
            value = value.lower() == "true"
        else:
            try:
                value = int(value)
            except ValueError:
                try:
                    value = float(value)
                except ValueError:
                    pass
        config[key] = value
    return config


def relative_path(path):
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def csv_missing_required_columns(path, required_columns):
    try:
        columns = pd.read_csv(path, nrows=0).columns
    except Exception as exc:
        return required_columns, str(exc)
    missing = [column for column in required_columns if column not in columns]
    return missing, None


def csv_contains_required_columns(path, required_columns):
    missing, error = csv_missing_required_columns(path, required_columns)
    return not missing and error is None


def newest_csv_in_folder(folder):
    if not folder.exists():
        return None
    csv_files = [path for path in folder.glob("*.csv") if path.is_file()]
    if not csv_files:
        return None
    return max(csv_files, key=lambda path: path.stat().st_mtime)


def newest_data_csv_with_required_columns(data_root, required_columns):
    if not data_root.exists():
        return None
    candidates = []
    for path in data_root.rglob("*.csv"):
        if path.is_file() and csv_contains_required_columns(path, required_columns):
            candidates.append(path)
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_mtime)


REQUIRED_COLUMNS = ["price_eur", "mileage_km", "age_years", "power_hp"]
SESSION_06_FULL_CSV = "autoscout24_listings_processed_audi_a3_germany_20251228_205210.csv"
OPTIONAL_COLUMNS = [
    "listing_id",
    "make",
    "model",
    "brand",
    "fuel_type",
    "listing_country",
    "registration_year",
    "registration_month",
    "price_label",
    "price_outlier_iqr",
    "mileage_outlier_iqr",
    "power_outlier_iqr",
    "logical_issue",
]


def select_processed_csv(required_columns, preferred_filename):
    preferred_paths = [
        ("Preferred full classroom sample CSV (local)", processed_folder / preferred_filename),
        ("Repo-visible fallback sample", sample_processed_folder / preferred_filename),
    ]
    rejected = []

    for source_label, candidate in preferred_paths:
        if candidate.exists():
            missing, error = csv_missing_required_columns(candidate, required_columns)
            if not missing and error is None:
                return candidate, source_label
            rejected.append((source_label, candidate, missing, error))

    candidate = newest_csv_in_folder(processed_folder)
    if candidate is not None:
        missing, error = csv_missing_required_columns(candidate, required_columns)
        if not missing and error is None:
            return candidate, "Latest local processed CSV"
        rejected.append(("Latest local processed CSV", candidate, missing, error))

    candidate = newest_data_csv_with_required_columns(PROJECT_ROOT / "data", required_columns)
    if candidate is not None:
        return candidate, "Last-resort scan for any CSV with required columns"

    details = []
    for source_label, path, missing, error in rejected:
        if error:
            details.append(f"{source_label}: {relative_path(path)} could not be read ({error})")
        else:
            details.append(f"{source_label}: {relative_path(path)} missing {missing}")
    raise FileNotFoundError(
        "No processed CSV with the required regression columns was found. "
        "Run Notebook 02 to create data/processed/*.csv, or use data/sample/processed/.\n"
        + "\n".join(details)
    )


def evaluate_regression_model(model_name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return {
        "model_name": model_name,
        "mae_test": mean_absolute_error(y_test, y_pred),
        "rmse_test": float(np.sqrt(mean_squared_error(y_test, y_pred))),
        "r2_test": r2_score(y_test, y_pred),
        "predictions": y_pred,
    }


def plot_actual_vs_predicted(y_true, y_pred, title):
    plt.figure(figsize=(6, 5))
    plt.scatter(y_true, y_pred, alpha=0.35)
    min_value = min(float(np.min(y_true)), float(np.min(y_pred)))
    max_value = max(float(np.max(y_true)), float(np.max(y_pred)))
    plt.plot([min_value, max_value], [min_value, max_value], color="black", linewidth=1)
    plt.title(title)
    plt.xlabel("Actual price (EUR)")
    plt.ylabel("Predicted price (EUR)")
    plt.tight_layout()
    plt.show()


def plot_residuals(y_true, y_pred, title):
    residuals = y_true - y_pred
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].hist(residuals, bins=30, color="#2563eb", alpha=0.8)
    axes[0].set_title("Residual distribution")
    axes[0].set_xlabel("Actual - predicted (EUR)")
    axes[0].set_ylabel("Listings")
    axes[1].scatter(y_pred, residuals, alpha=0.35)
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set_title("Residuals vs predicted price")
    axes[1].set_xlabel("Predicted price (EUR)")
    axes[1].set_ylabel("Residual (EUR)")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_price_relationship(data, feature, xlabel):
    plt.figure(figsize=(7, 4.5))
    plt.scatter(data[feature], data["price_eur"], alpha=0.35)
    plt.title(f"Price vs {xlabel}")
    plt.xlabel(xlabel)
    plt.ylabel("Price (EUR)")
    plt.tight_layout()
    plt.show()


PROJECT_ROOT = find_repo_root(Path.cwd())
PROJECT_CONFIG = load_project_config(PROJECT_ROOT / "config" / "project_config.yaml")
RANDOM_SEED = int(PROJECT_CONFIG.get("random_state", 42))
np.random.seed(RANDOM_SEED)

make_scope = str(PROJECT_CONFIG.get("make", "Audi")).strip()
model_scope = str(PROJECT_CONFIG.get("model", "A3")).strip()
country_scope = str(PROJECT_CONFIG.get("country", "Germany")).strip()
TAG = f"{make_scope}_{model_scope}_{country_scope}".lower().replace(" ", "_")

processed_folder = PROJECT_ROOT / str(PROJECT_CONFIG.get("processed_data_path", "data/processed"))
sample_processed_folder = PROJECT_ROOT / "data" / "sample" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

selected_csv_path, selected_source = select_processed_csv(REQUIRED_COLUMNS, SESSION_06_FULL_CSV)
df_raw = pd.read_csv(selected_csv_path)
row_count_before = len(df_raw)
missing_required = [column for column in REQUIRED_COLUMNS if column not in df_raw.columns]
if missing_required:
    raise ValueError("Missing required regression columns: " + ", ".join(missing_required))

missing_optional = [column for column in OPTIONAL_COLUMNS if column not in df_raw.columns]
df = df_raw.copy()
for column in REQUIRED_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors="coerce")

valid_modeling_rows = (
    df["price_eur"].gt(0)
    & df["mileage_km"].gt(0)
    & df["age_years"].ge(0)
    & df["power_hp"].gt(0)
)
rows_removed = int((~valid_modeling_rows).sum())
df = df.loc[valid_modeling_rows].reset_index(drop=True)
if "listing_id" not in df.columns:
    df.insert(0, "listing_id", np.arange(1, len(df) + 1))
    missing_optional = [column for column in missing_optional if column != "listing_id"]

if df.empty:
    raise ValueError("The selected CSV has no valid modeling rows after filtering.")

print("Selected CSV:", relative_path(selected_csv_path))
print("Source priority:", selected_source)
print("Configured scope:", make_scope, model_scope, country_scope)
print("Rows before filtering:", row_count_before)
print("Rows after filtering:", len(df))
print("Rows removed by required-value filter:", rows_removed)
print("Available columns:")
print(", ".join(df.columns))
if missing_optional:
    print("Optional columns not found in this CSV:", ", ".join(missing_optional))

print("\nFirst 5 rows:")
display(df.head())

print("Missing values summary:")
missing_summary = df.isna().sum().sort_values(ascending=False).to_frame("missing_values")
display(missing_summary[missing_summary["missing_values"] > 0])


## Desafío 1: lectura visual del mercado antes de modelar

**Tiempo sugerido:** 20 minutos

**Tarea del estudiante**

**Razón comercial:**  
Un científico de datos debería leer el mercado visualmente antes de ajustar un modelo. Los gráficos revelan relaciones, valores atípicos, no linealidad y posibles problemas de datos que las métricas por sí solas pueden ocultar.

**Objetivo:**  
Inspeccionar la relación entre el precio y las principales variables explicativas.

**Formato de salida esperado:**
- Tres diagramas de dispersión.
- Tres breves ideas visuales.
- Una hipótesis sobre el predictor más fuerte.


In [ ]:
# Ayudante de trazado dirigido por un instructor; La tarea del estudiante es la interpretación.
plot_df = df.sample(n=min(20000, len(df)), random_state=RANDOM_SEED)

plot_price_relationship(plot_df, "mileage_km", "mileage (km)")
plot_price_relationship(plot_df, "age_years", "age (years)")
plot_price_relationship(plot_df, "power_hp", "power (hp)")


### Tus conocimientos visuales

1. Información sobre el kilometraje:
2. Información sobre la edad:
3. Conocimiento de poder:
4. Hipótesis sobre el predictor más fuerte:


## Desafío 2: un modelo de referencia simple

**Tiempo sugerido:** 30 minutos

**Tarea del estudiante**

**Razón comercial:**  
Antes de construir un modelo expected-price serio, necesitamos una línea de base simple. Un modelo más complejo debe superar una regla simple y explicable.

**Objetivo:**  
Entrene un modelo de referencia: `price_eur ~ age_years`.

**Formato de salida esperado:**
- Métricas base: MAE, RMSE, R2.
- Coeficiente e intercepto.
- Una interpretación de coeficientes en inglés sencillo.

**Pausa de discusión:**
- ¿Qué significa MAE en euros?
- ¿Tiene sentido comercial el signo del coeficiente?
- ¿Es suficiente una variable para apoyar la decisión?


In [ ]:
# Desafío 2 andamio.
# Entrene un modelo de regresión lineal simple: precio_eur ~ age_years.

# Tu código aquí
# train_df, test_df = train_test_split(df, test_size=0.20, random_state=RANDOM_SEED)
# línea_base_features = ["edad_años"]
# X_tren = tren_df[características_base]
# X_test = test_df[base_features]
# y_tren = tren_df["precio_eur"]
# y_test = test_df["precio_eur"]
#
# baseline_model = Regresión lineal()
# resultado_linea base = evaluar_modelo_regresión(
#     "Línea de base lineal solo por edad",
#     modelo_de línea base,
#     tren_x,
#     X_prueba,
#     y_tren,
#     y_prueba,
# )
#
# Construya una tabla métrica de una fila e interprete el coeficiente.


### Su conclusión básica

1. Variable de referencia utilizada:
2. MAE en lenguaje empresarial:
3. Interpretación del coeficiente:
4. ¿Es esto suficiente para respaldar las decisiones?


## Desafío 3: modelo multivariado expected-price

**Tiempo sugerido:** 40 minutos

**Tarea del estudiante con funciones de ayuda**

**Razón comercial:**  
Una capa útil expected-price debería controlar varias características del vehículo al mismo tiempo. El kilometraje, la edad y la potencia explican cada uno parte del mercado.

**Objetivo:**  
Entrene `price_eur ~ mileage_km + age_years + power_hp` y compárelo con la línea base simple.

**Formato de salida esperado:**
- Tabla de métricas que comparan el modelo basal y multivariado.
- Tabla de coeficientes.
- Trama real versus prevista.
- Parcela residual.

**Pausa de discusión:**
- ¿El modelo mejoró el error del euro?
- ¿Tienen sentido económico los signos de los coeficientes?
- ¿Dónde sugieren los residuos que el modelo es débil?


In [ ]:
# Desafío 3 andamio.
# Utilice las funciones auxiliares para evitar repetir mecánicas métricas y de trazado.

# Tu código aquí
# core_features = ["mileage_km", "age_years", "power_hp"]
# multivariate_model = Regresión lineal()
# resultado_multivariado = evaluar_modelo_regresión(
#     "Regresión lineal multivariada central",
#     modelo_multivariado,
#     train_df[características_principales],
#     test_df[características_principales],
#     y_tren,
#     y_prueba,
# )
#
# Compare baseline_metrics_df con las métricas multivariadas.
# Cree una tabla de coeficientes a partir de multivariate_model.coef_.
# Utilice plot_actual_vs_predicted(...) y plot_residuals(...).


### Tu conclusión multivariante

1. ¿Mejoró el modelo multivariado MAE?
2. ¿Qué coeficiente es más fácil de explicar?
3. ¿Dónde parece débil el modelo?
4. ¿Es lo suficientemente útil como capa de referencia expected-price?


## Desafío 4: del resultado de regresión a la cola de decisiones

**Tiempo sugerido:** 35 minutos

**Tarea del estudiante**

**Razón comercial:**  
Un modelo de regresión resulta útil cuando produce un artefacto de apoyo a las decisiones. La capa expected-price debería ayudar a los analistas a priorizar qué listados merecen revisión.

**Objetivo:**  
Entrene el modelo seleccionado en todas las filas válidas, genere resultados expected-price y cree una cola de revisión.

**Formato de salida esperado:**
-`expected_price_eur`
- `expected_price_gap`
- `expected_price_gap_pct`
- Principales candidatos más baratos de lo esperado.
- Principales candidatos más caros de lo esperado.
- Cola de revisión guardada en `reports/{TAG}_regression_review_queue.csv`.

**Pausa de discusión:**
- ¿Qué 3 listados merecen la revisión de los analistas primero?
- ¿Qué listados pueden ser artefactos modelo?
- ¿Qué debería mostrar BI y qué debería evitar implicar?


In [ ]:
# Desafío 4 andamio.
# El modelo seleccionado puede ser el modelo multivariado central a menos que tenga una razón clara para elegir lo contrario.

# Tu código aquí
# selected_model_name = "Regresión lineal multivariada central"
# características_seleccionadas = ["kilómetros_km", "años_edad", "potencia_hp"]
# modelo_seleccionado = Regresión lineal()
# modelo_seleccionado.fit(df[características_seleccionadas], df["precio_eur"])
# expected_price = modelo_seleccionado.predict(df[características_seleccionadas])
#
# review_queue = df[["listing_id", "precio_eur", "mileage_km", "age_years", "power_hp"]].copiar()
# cola_revisión["precio_expectado_eur"] = expected_price
# cola_revisión["precio_esperado_gap"] = cola_revisión["precio_eur"] - cola_revisión["expected_price_eur"]
# review_queue["expected_price_gap_pct"] = review_queue["expected_price_gap"] / review_queue["expected_price_eur"]
#
# Ordene para inspeccionar listados más baratos y más caros de lo esperado.
# Guardar en REPORTS_DIR / f"{TAG}_regression_review_queue.csv".


### La conclusión de tu cola de revisión

1. Tres listados para revisión de analistas:
2. Limitación de un modelo:
3. Una próxima mejora:
4. ¿Qué debería mostrar BI?


## Extensión opcional: modelos flexibles

**Extensión opcional**

Úselo solo si la clase avanza más rápido de lo esperado o como práctica después de clase.

**Demostración dirigida por un instructor**

Los modelos flexibles pueden capturar relaciones no lineales, pero pueden ser más difíciles de explicar. Compare el modelo lineal central con:

-`DecisionTreeRegressor(max_depth=4, random_state=RANDOM_SEED)`
- `RandomForestRegressor(n_estimators=100, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1)`
- `HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=31, random_state=RANDOM_SEED)`

Discusión:
- ¿Es la mejora lo suficientemente grande como para importar en euros?
- ¿Qué modelo sería más fácil de defender ante un comité?
- ¿Qué podría salir mal si sólo perseguimos un error menor?


In [ ]:
# Andamio de extensión opcional.
# Ejecute esto después del laboratorio central sólo si hay tiempo.

# Tu código aquí
# modelos_avanzados = {
#     "Árbol de decisión": DecisionTreeRegressor (max_profundidad = 4, random_state = RANDOM_SEED),
#     "Bosque aleatorio": RandomForestRegressor(n_estimators=100, max_profundidad=8, random_state=RANDOM_SEED, n_jobs=-1),
#     "Aumento del gradiente de histograma": HistGradientBoostingRegressor (max_iter=200, learning_rate=0.05, max_leaf_nodes=31, random_state=RANDOM_SEED),
# }
# Compare estos modelos con el modelo multivariado central utilizando evalua_regression_model(...).


## Discusión final: ¿qué aprendimos sobre la regresión como capa de apoyo a las decisiones?

Utilice estas indicaciones para la discusión final en clase:

- ¿Por qué inspeccionamos visualmente el mercado antes de modelar?
- ¿Por qué construimos una línea de base simple?
- ¿Cuál es el significado comercial de MAE, RMSE y R2?
- ¿Cómo se convierten los residuos en posibles señales de fijación de precios incorrectos?
- ¿Por qué `expected_price_gap` es una señal de priorización y no una decisión?
- ¿Cómo debería combinar BI el precio real y la brecha expected-price?
- ¿Por qué el resultado del modelo todavía requiere el juicio de los analistas?

Mensaje final:

**Un modelo de regresión útil no termina en una métrica. Termina en una capa de referencia que ayuda a las personas a explicar las desviaciones, priorizar la revisión y tomar mejores decisiones.**
